# AB-MEDAL and AB-MEDAL + AdaLook: paired experiments

[Open in Colab](https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/chong/notebooks/gsm8k/AB_MEDAL_MCTS_Colab.ipynb)

Default `comparison` mode compares **AB-MEDAL (IG)** with **AB-MEDAL + adaptive AdaLook** on five frozen development questions, Lc=10, search budget=256, batch size=1. It loads one model and saves every question, paired changes, CSV/Markdown summaries and a comparison chart. Add `adalook_one_step` to isolate the reward change from lookahead.

`demo` mode inspects one question: real search tree, Student-t GEN/CONT decisions, posterior updates, temporary comparison hypotheses and rollout NFEs. AdaLook here is a project hybrid, not a reproduction of the paper decoder. Temporary candidates never become real AB nodes. All extra forwards consume the search budget.

This notebook requires a CUDA/bfloat16 runtime for inference. Editing it does not run experiments. In Colab, setup checks out `chong`; push the implementation there before using that setup. See [the implementation guide](../../docs/ab_medal_walkthrough.md).


## Setup — once per runtime

Use the pinned LLaDA dependency profile and a CUDA runtime with bfloat16 support. The existing checkout and Drive cache flow is preserved. Validate the frozen manifest before loading weights. All later cells reuse one model.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "chong"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError(f"The checkout differs from origin/{BRANCH}. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local checkout: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM") if IN_COLAB else REPO_DIR / ".cache/medal"
os.environ.update(
    HF_HUB_CACHE=str(DRIVE_ROOT / "cache/hub"),
    HF_DATASETS_CACHE=str(DRIVE_ROOT / "cache/datasets"),
    HF_HUB_OFFLINE="0", HF_DATASETS_OFFLINE="0", TRANSFORMERS_OFFLINE="0",
    TOKENIZERS_PARALLELISM="false",
)
for key in ("TRANSFORMERS_CACHE", "PYTORCH_TRANSFORMERS_CACHE", "PYTORCH_PRETRAINED_BERT_CACHE"):
    os.environ.pop(key, None)

In [ ]:
import json
import torch
from IPython.display import Image, Markdown, display
from dlm_project.models.profiles import get_profile
from dlm_project.experiments.paths import create_run_directory
from dlm_project.decoding.medal_demo import show_candidates
from dlm_project.decoding import search_audit
from dlm_project.tasks.gsm8k import ab_development as dev, ab_comparison
from dlm_project.tasks.gsm8k.ab_batch_study import run_study

NOTEBOOK_MODE = "comparison"  # comparison | demo
METHOD = "ab_medal"
STAGE = "dev"  # dev (5 available) | evaluation (20) | test (200)
NUM_SAMPLES = 5
CONDITION = "A"  # A=(Lc=10, budget=256), B=(Lc=10, budget=512)
BATCH_SIZE = 1
RESUME_RUN = None  # Previous comparison directory; retain settings and code.
COPY_MODEL_TO_RUNTIME = True
COMPARISON_REWARD_MODES = ["ig", "adalook_adaptive"]
# Optional scoring-only ablation: add "adalook_one_step" above.
REWARD_MODE = "adalook_adaptive"  # Used by demo mode.
SAMPLE_INDEX = 0  # Used by demo mode, from the dev pool.
ADALOOK_OPTIONS = dict(adalook_alpha=0.1, adalook_confidence_threshold=0.8,
    adalook_max_depth=2, adalook_variance_threshold=0.01,
    adalook_trigger_threshold=0.2, adalook_min_masks=2)
# These thresholds are experimental starting values, not tuned results.
REWARD_OPTIONS = dict(ADALOOK_OPTIONS, reward_mode=REWARD_MODE)
if NOTEBOOK_MODE not in ("comparison", "demo"):
    raise ValueError("NOTEBOOK_MODE must be comparison or demo.")
if BATCH_SIZE != 1:
    raise ValueError("Paired comparisons use serial execution; batching has a separate validation section.")
VARIANTS = None
if NOTEBOOK_MODE == "comparison":
    VARIANTS = ab_comparison.contexts(stage=STAGE, condition=CONDITION,
        num_samples=NUM_SAMPLES, reward_modes=COMPARISON_REWARD_MODES,
        adalook_options=ADALOOK_OPTIONS)
    ctx = VARIANTS[0]
else:
    ctx = dev.context(stage="dev", condition=CONDITION, indices=(SAMPLE_INDEX,),
                      method=METHOD, reward_options=REWARD_OPTIONS)
SEARCH_BUDGET = ctx["search"]["max_model_calls"]
print("Mode:", NOTEBOOK_MODE, "| stage:", ctx["stage"])
print("Frozen questions:", [e["example_id"] for e in ctx["examples"]])
print("Shared protocol:", ctx["common_protocol_hash"])
for variant in VARIANTS or [ctx]:
    print("Search settings:", variant["search"], "| variant:", variant["variant_hash"])
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Select a CUDA runtime supporting the shared bfloat16 model configuration.")
print("GPU:", torch.cuda.get_device_name(0))
SINGLE_OUTPUT = None
if NOTEBOOK_MODE == "demo":
    SINGLE_OUTPUT, _ = create_run_directory({"output_dir": str(DRIVE_ROOT / "results/ab_medal/development")})


## Download / reuse the pinned model

Reuse the same snapshot cache as MEDAL. In Colab, optionally stage weights from Drive to runtime disk. Model loading and warm-up occur once, before timed decoding.


In [ ]:
from dlm_project.models.cache import cache_model

model_name = get_profile("llada")["repo_id"].split("/")[-1]
runtime_models = Path("/content/models") if IN_COLAB else REPO_DIR / ".cache/models"
MODEL_PATH = str(cache_model(
    drive_dir=DRIVE_ROOT / "models" / model_name,
    runtime_dir=runtime_models / model_name if COPY_MODEL_TO_RUNTIME else None,
    family="llada",
))
print("Pinned model ready:", MODEL_PATH)

In [ ]:
llada = dev.load_model(ctx, model_path=MODEL_PATH, device="cuda", local_files_only=True)
prepared = dev.prepare(llada, ctx)
item = prepared[0]
ctx["setup"]["warmup"] = dev.warmup(llada, prepared, ctx)
print("Setup measurements (excluded from decoding):", ctx["setup"])


## Run the paired comparison

In `comparison` mode, evaluate each reward variant sequentially on the same frozen questions and seeds. The runner uses `ABMEDALSearch` for every variant. Results are saved under `MyDrive/CSCI544-DLM/results/ab_medal/comparison/<stage>/<condition>/<run>/`.

Outputs: `predictions.jsonl`, per-attempt `traces/`, complete per-question artifacts in `questions/`, `summary.json`, `report/summary.csv`, `report/per_question.csv`, `report/paired_results.csv`, `report/report.md`, and `report/comparison.png`. Accuracy covers completed questions; paired changes cover only questions completed in both variants. The configuration records shared inputs, all priors/reward settings and an implementation fingerprint.

To resume, set `RESUME_RUN` to the printed directory, keep settings/code unchanged and rerun setup. Completed variant/question pairs are skipped; an interrupted question restarts. To redisplay saved results, set `COMPARISON_OUTPUT = Path("...")` and run only the display cell. Reports can be viewed without loading the model.


In [ ]:
COMPARISON_OUTPUT = None
if NOTEBOOK_MODE == "comparison":
    COMPARISON_OUTPUT = ab_comparison.run_comparison(VARIANTS,
        output_root=DRIVE_ROOT / "results/ab_medal/comparison",
        resume=RESUME_RUN, wrapper=llada)
    print("Comparison artifacts:", COMPARISON_OUTPUT)


In [ ]:
if COMPARISON_OUTPUT is not None:
    display(Markdown((COMPARISON_OUTPUT / "report/report.md").read_text()))
    plot_path = COMPARISON_OUTPUT / "report/comparison.png"
    if plot_path.is_file():
        display(Image(filename=str(plot_path)))
    print("Predictions:", COMPARISON_OUTPUT / "predictions.jsonl")
    print("Summary CSV:", COMPARISON_OUTPUT / "report/summary.csv")
    print("Paired changes:", COMPARISON_OUTPUT / "report/paired_results.csv")


## Optional single-question walkthrough

All remaining inference cells run only in `demo` mode. Set `NOTEBOOK_MODE="demo"`, select `REWARD_MODE` and `SAMPLE_INDEX`, and rerun from settings. The demo always uses the development pool. Optional serial and batch studies remain disabled by default; enable batching only after serial validation.


## 1. Prompting and initial state

The shared pens/muffins two-shot template is passed through the pinned tokenizer's existing user chat format. No demonstrations are fetched and no model forward generates a plan. The current reference answer is used only for scoring. Search starts with these exact prompt IDs followed by 256 mask tokens.

In [ ]:
if NOTEBOOK_MODE == "demo":
    print(item["text"])
    print("Template hash:", item["prompt_template_hash"])
    print("Rendered-prompt hash:", item["prompt_hash"])
    print("Token-ID hash:", item["prompt_ids_hash"])
    print("Prompt tokens:", len(item["prompt_ids"]), "| output masks: 256 | seed:", item["seed"])

## 2. AB search and optional AdaLook evaluation

Each traversal starts at the root. GEN chooses the next retained action; CONT samples eligible child predictions. Each iteration expands one real child. Priors remain `(m, kappa, nu, tau2) = (0, 1, 1, 0.1)` and predictive draws remain Student-t.

`ig` uses `(H(parent)-H(child))/H(parent)` (zero for negligible parent entropy); negative rewards are retained. `adalook_one_step` uses the current confidence score without extra forwards. `adalook_adaptive` compares the real child to detached hypotheses for other retained parent actions, reuses existing real-sibling base analyses, and refines temporary trajectories through the shared AdaLook evaluator. Only the real child's final score is backed up, exactly once, including its GEN/CONT posterior recipients. Pruned origins keep their last valid score.

Temporary hypotheses never consume real actions, enter the tree, or change final-completion commitments. Root, real-child, temporary-comparison and rollout forwards all consume the search budget. A complete comparison group or refinement round must fit before the next budget checkpoint; otherwise use the last valid score. Search stops at the budget or exhausted frontier, not at three candidates.


In [ ]:
if NOTEBOOK_MODE == "demo":
    initialization = dev.initialize(llada, item, ctx, SINGLE_OUTPUT)
    completion = response = record = None
    print("Stop:", initialization.search.stop_reason)
    print("Search NFEs:", initialization.result.stats["search_model_calls"])
    print("Selected depth:", initialization.result.selected.depth)

### Candidates and final shortlist

Choose the deepest evaluated depth with at least C=3 distinct states. If none qualifies, choose the depth with the most distinct states, preferring deeper on ties. Deduplicate by first occurrence, rank by empirical `Q = reward_sum / visits` (ascending node ID for ties), retain at most three and complete rank 1. Temporary AdaLook hypotheses and posterior samples do not rank candidates.

Fallback means selected depth below requested depth. One/two-item shortlists and the analyzed root are valid. The root has zero visits and Q=0. Requested-depth candidate count is reported separately.


In [ ]:
if NOTEBOOK_MODE == "demo":
    show_candidates(initialization, tokenizer=llada.tokenizer)
    print("Candidates at selected depth:", initialization.result.stats["candidate_count"])
    print("Requested-depth candidates:", initialization.result.stats["target_depth_candidate_count"])
    print("Shortlist:", initialization.result.stats["candidate_shortlist"])
    print("Fallback:", initialization.result.stats["fallback_decode_used"])
    ranked_candidates = sorted(initialization.result.stats["candidate_evaluations"],
                               key=lambda c: (-c["q_value"], c["node_id"]))
    candidate_table = ["| Rank | Node | Depth | Visits | Reward sum | Immediate IG | Empirical Q |",
                       "| --- | --- | --- | --- | --- | --- | --- |"]
    for rank, candidate in enumerate(ranked_candidates, 1):
        candidate_table.append("| " + " | ".join(str(v) for v in (
            rank, candidate["node_id"], candidate["depth"], candidate["visits"],
            candidate["reward_sum"], candidate["immediate_gain"], candidate["q_value"])) + " |")
    display(Markdown("\n".join(candidate_table)))

    print("Reward mode:", ctx["search"]["reward_mode"])
    print("AdaLook groups:", initialization.result.stats["adalook_groups"])


## 3. LLaDA Confidence completion and scoring

Use temperature 0 and preserve every prompt/search commitment. Each forward fills one remaining output mask, so completion NFEs equal `256 - selected_depth`. EOS/EOT are allowed during completion and do not stop fixed-length generation. Score only the completed output prefix before the first terminator, using strict `####` parsing.

In [ ]:
if NOTEBOOK_MODE == "demo":
    completion = dev.complete(llada, item, initialization, ctx, SINGLE_OUTPUT)
    record = dev.finish_single(llada, item, ctx, initialization, completion, SINGLE_OUTPUT)
    response = record["decoded"]["scoring_text"]
    print(response)
    print("Score:", record["score"])
    print("Artifacts:", SINGLE_OUTPUT)

## 4. All metrics for n=1, reconstructed offline

This includes accuracy/parsing; maximum/selected/mean depth; branching min/mean/max and expanded parents; deepest/target candidates and unique states; negative IG count/percentage, IG distribution and selected Q; NFEs per depth and root subtree; GEN/CONT and forced decisions; duplicates, target success, fallback, and costs.

Branching counts evaluated children per expanded parent. Root analysis is depth 0 and is separate from subtree allocation. Negative IG and duplicate percentages use evaluated children; GEN/CONT percentages use decisions, because one child evaluation may involve multiple CONT decisions. Timing excludes setup, warm-up, tokenization, scoring, offline analysis, plots, and artifact writes.

In [ ]:
if NOTEBOOK_MODE == "demo":
    offline_metrics = search_audit.load(SINGLE_OUTPUT)
    for key, value in offline_metrics.items():
        assert record[key] == value, f"Offline mismatch: {key}"
    extra_nfes = offline_metrics["adalook_total_extra_nfes"]
    assert sum(offline_metrics["nfes_per_depth"].values()) + extra_nfes == record["search_nfes"]
    assert offline_metrics["root_analysis_nfes"] + sum(offline_metrics["nfes_per_root_subtree"].values()) + extra_nfes == record["search_nfes"]
    assert record["total_nfes"] == record["search_nfes"] + record["completion_nfes"]
    assert record["completion_nfes"] == 256 - record["selected_depth"]
    dev.show_metrics(record)
    dev.plot_metrics(SINGLE_OUTPUT)
    display(Image(filename=str(SINGLE_OUTPUT / "compute_allocation.png")))
    display(Image(filename=str(SINGLE_OUTPUT / "ab_medal_tree.png")))
    for key, label in (("nfes_per_depth", "Depth"), ("nfes_per_root_subtree", "Root-child node")):
        table = [f"| {label} | Search sequence NFEs |", "| --- | --- |"]
        table.extend(f"| {identifier} | {nfes} |" for identifier, nfes in offline_metrics[key].items())
        display(Markdown("\n".join(table)))
    print("Root-analysis NFEs (outside subtrees):", offline_metrics["root_analysis_nfes"])


## 5. Optional serial validation on all five development questions

Enable after inspecting the first run. This keeps the selected depth/budget, uses all five frozen development questions in manifest order, and reuses the loaded model. Every question gets independent search statistics and its fixed seed. Results are flushed after each question; interruptions and errors preserve partial summaries.

In [ ]:
RUN_SERIAL_DEV = False
serial_development = None
if NOTEBOOK_MODE == "demo" and RUN_SERIAL_DEV:
    dev_ctx = dev.context(method=METHOD, reward_options=REWARD_OPTIONS, stage="dev", condition=CONDITION, indices=tuple(range(5)))
    dev_inputs = dev.prepare(llada, dev_ctx)
    dev_output, _ = create_run_directory({"output_dir": str(DRIVE_ROOT / "results/ab_medal/dev5")})
    serial_development = dev.run_samples(llada, dev_inputs, dev_ctx, dev_output, batch_size=1)
    print(json.dumps(serial_development["summary"], indent=2))

## 6. Optional batch validation, then throughput at sizes 1 / 2 / 4

Enable only after successful five-question serial validation. Each batch contains independent questions, not siblings from one tree. The same first four development questions and fixed seeds are reused for every size. Search NFEs remain per question; actual model invocations and batch occupancy are reported separately.

Before timing, compare unpadded serial, padded serial, and batched predictions on saved root/selected states; then compare complete searches and answers. Discrete decisions, selected states, and output tokens must agree. IG/Q use `rtol=1e-4, atol=1e-5`. Attention masks alone are not a parity guarantee. Failed configurations are excluded from throughput results.

Three repetitions rotate batch-size order after separate warm-up. Reports include runtime spread, throughput/speedup, memory, occupancy, NFEs, correctness and fallback. OOMs are recorded without reducing batch size. Group wall time is used for speedup; per-question latencies overlap and must not be summed. Future MEDAL/AB algorithm comparisons must use matched execution conditions; serial remains the default.

In [ ]:
RUN_BATCH_STUDY = False
REPETITIONS = 3
batch_study = None
if NOTEBOOK_MODE == "demo" and RUN_BATCH_STUDY:
    if serial_development is None or serial_development["summary"]["status"] != "completed":
        raise RuntimeError("Complete the optional five-question serial validation section first.")
    batch_ctx = dev.context(method=METHOD, reward_options=REWARD_OPTIONS, stage="dev", condition=CONDITION, indices=tuple(range(4)))
    batch_inputs = dev.prepare(llada, batch_ctx)
    batch_output, _ = create_run_directory({"output_dir": str(DRIVE_ROOT / "results/ab_medal/batching")})
    batch_study = run_study(llada, batch_inputs, batch_ctx, batch_output, repetitions=REPETITIONS)
    print(json.dumps(batch_study["summary"], indent=2))
    print("Validation:", json.dumps(batch_study["validation"], indent=2))
    if (batch_output / "batch_comparison.png").is_file():
        display(Image(filename=str(batch_output / "batch_comparison.png")))
    print("Batch artifacts:", batch_output)